# Lab 1 — Outlier Exploration

**Day 06 · Anomaly Detection · Cisco AI/ML Training**

---

## Goals

1. Load the credit card transaction dataset (**1,000** rows, **10** fraud).
2. Count **IQR outliers** in `amount` and `distance_from_home`.
3. Compare fraud vs legitimate transaction profiles.
4. Explain why rule-based outlier flags alone are insufficient for fraud.

> **Quick check:** **1000** rows · **10** fraud · mean fraud amount ≈ **234** · legit ≈ **43**

**Dataset:** `data/credit-card/credit_card_transactions.csv`

## Outliers vs fraud

Fraudulent transactions often look like **outliers**, but not every outlier is fraud.

## Anomaly detection methods (course syllabus)

| Method | Type | This course |
|--------|------|-------------|
| IQR / rules | Statistical | **This lab** |
| LOF | Proximity | Day 6 Lab 4 |
| Random Forest | Ensemble supervised | Day 6 Lab 5 |

In [ ]:
%matplotlib inline

from pathlib import Path

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

df = pd.read_csv(GH_ROOT / "data" / "credit-card" / "credit_card_transactions.csv")

print("Lab 1 — Outlier exploration")
print(f"rows: {len(df)}")
print(f"fraud rows: {int(df['is_fraud'].sum())}")
print(f"fraud rate: {df['is_fraud'].mean():.4f}")
display(df.head(3))


## IQR outlier helper

In [ ]:
def iqr_outlier_count(series: pd.Series) -> int:
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((series < lower) | (series > upper)).sum())

def iqr_bounds(series: pd.Series) -> tuple[float, float]:
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

amount_outliers = iqr_outlier_count(df["amount"])
distance_outliers = iqr_outlier_count(df["distance_from_home"])

print(f"IQR outliers (amount): {amount_outliers}")
print(f"IQR outliers (distance): {distance_outliers}")



## Fraud vs legitimate means

In [ ]:
legit_amount_mean = df.loc[df["is_fraud"] == 0, "amount"].mean()
fraud_amount_mean = df.loc[df["is_fraud"] == 1, "amount"].mean()
max_fraud_distance = df.loc[df["is_fraud"] == 1, "distance_from_home"].max()

print(f"mean amount (legit): {legit_amount_mean:.2f}")
print(f"mean amount (fraud): {fraud_amount_mean:.2f}")
print(f"max distance (fraud): {max_fraud_distance:.2f}")

compare = pd.DataFrame({
    "group": ["legit", "fraud"],
    "mean_amount": [legit_amount_mean, fraud_amount_mean],
    "count": [(df["is_fraud"] == 0).sum(), (df["is_fraud"] == 1).sum()],
})
display(compare.round(2))


## Prediction outliers (course topic)




In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X_pred = df[["amount", "distance_from_home"]]
y_true = df["is_fraud"]

clf = Pipeline(
    [
        ("scale", StandardScaler()),
        ("lr", LogisticRegression(max_iter=500, random_state=42)),
    ]
)
clf.fit(X_pred, y_true)
proba = clf.predict_proba(X_pred)[:, 1]
residual = np.abs(y_true.to_numpy() - proba)

cutoff = float(np.quantile(residual, 0.99))
pred_outlier_mask = residual >= cutoff
n_pred_outliers = int(pred_outlier_mask.sum())
fraud_in_pred_outliers = int(df.loc[pred_outlier_mask, "is_fraud"].sum())

print(f"prediction-outlier threshold (99th pct |y - p|): {cutoff:.4f}")
print(f"flagged rows: {n_pred_outliers}")
print(f"fraud among flagged: {fraud_in_pred_outliers} / {int(y_true.sum())} total fraud")


## Visualize amount and distance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.boxplot(data=df, x="is_fraud", y="amount", ax=axes[0], palette="Set2")
axes[0].set_title("Transaction amount by fraud label")
axes[0].set_xticklabels(["legit (0)", "fraud (1)"])

sns.boxplot(data=df, x="is_fraud", y="distance_from_home", ax=axes[1], palette="Set2")
axes[1].set_title("Distance from home by fraud label")
axes[1].set_xticklabels(["legit (0)", "fraud (1)"])

plt.tight_layout()
plt.show()


## Fraud transactions

In [ ]:
fraud_df = df.loc[df["is_fraud"] == 1].sort_values("amount", ascending=False)
display(fraud_df[["amount", "distance_from_home", "merchant_category"]].round(2))


## Amounts above IQR upper bound

In [ ]:
lower, upper = iqr_bounds(df["amount"])
high_amount = df.loc[df["amount"] > upper, ["amount", "distance_from_home", "is_fraud", "merchant_category"]]

print(f"IQR upper bound (amount): {upper:.2f}")
print(f"rows above upper bound: {len(high_amount)}")
print(f"fraud among high-amount rows: {int(high_amount['is_fraud'].sum())}")
display(high_amount.head(8).round(2))


### Outlier EDA prompt 1

Show column dtypes.

In [ ]:
print(df.dtypes)

### Outlier EDA prompt 2

Count merchant categories.

In [ ]:
print(df['merchant_category'].value_counts().head(8))

### Outlier EDA prompt 3

Compute median amount by fraud label.

In [ ]:
print(df.groupby('is_fraud')['amount'].median().round(2).to_dict())

### Outlier EDA prompt 4

Compute std amount by fraud label.

In [ ]:
print(df.groupby('is_fraud')['amount'].std().round(2).to_dict())

### Outlier EDA prompt 5

Show amount percentiles.

In [ ]:
print(df['amount'].quantile([0.25, 0.5, 0.75, 0.99]).round(2).to_dict())

### Outlier EDA prompt 6

Show distance percentiles.

In [ ]:
print(df['distance_from_home'].quantile([0.25, 0.5, 0.75, 0.99]).round(2).to_dict())

### Outlier EDA prompt 7

Count fraud by merchant category.

In [ ]:
print(df.groupby('merchant_category')['is_fraud'].sum().sort_values(ascending=False).head(5))

### Outlier EDA prompt 8

Scatter amount vs distance colored by fraud.

In [ ]:
ax = df.plot.scatter(x='amount', y='distance_from_home', c=df['is_fraud'], cmap='coolwarm', figsize=(6,4), alpha=0.6); ax.set_title('Amount vs distance');

### Outlier EDA prompt 9

List top 5 amounts overall.

In [ ]:
display(df.nlargest(5, 'amount')[['amount','distance_from_home','is_fraud']].round(2))

### Outlier EDA prompt 10

List top 5 distances overall.

In [ ]:
display(df.nlargest(5, 'distance_from_home')[['amount','distance_from_home','is_fraud']].round(2))

### Outlier EDA prompt 11

Compute fraud rate by merchant category.

In [ ]:
rates = df.groupby('merchant_category')['is_fraud'].mean().sort_values(ascending=False); print(rates.head(5).round(4))

### Outlier EDA prompt 12

Check missing values.

In [ ]:
print(df.isna().sum().to_dict())

### Outlier EDA prompt 13

Histogram of amount.

In [ ]:
ax = df['amount'].plot(kind='hist', bins=30, figsize=(5,3)); ax.set_title('Amount distribution');

### Outlier EDA prompt 14

Histogram of distance.

In [ ]:
ax = df['distance_from_home'].plot(kind='hist', bins=30, figsize=(5,3)); ax.set_title('Distance distribution');

### Outlier EDA prompt 15

Compare IQR bounds for amount.

In [ ]:
lo, hi = iqr_bounds(df['amount']); print({'lower': round(lo,2), 'upper': round(hi,2)})

### Outlier EDA prompt 16

Count rows below amount lower bound.

In [ ]:
lo, hi = iqr_bounds(df['amount']); print(int((df['amount'] < lo).sum()))

### Outlier EDA prompt 17

Summarize fraud amount range.

In [ ]:
print(df.loc[df['is_fraud']==1, 'amount'].agg(['min','max','mean']).round(2).to_dict())

### Outlier EDA prompt 18

Summarize legit amount range.

In [ ]:
print(df.loc[df['is_fraud']==0, 'amount'].agg(['min','max','mean']).round(2).to_dict())

### Outlier EDA prompt 19

Re-state dataset size.

In [ ]:
print({'rows': len(df), 'fraud': int(df['is_fraud'].sum())})

### Outlier EDA prompt 20

Bridge to imbalance lab.

In [ ]:
print('Next: quantify 99:1 imbalance and baseline accuracy trap.')

### Lab 1 quick recap 1

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 1 recap step 1: completed")

### Lab 1 quick recap 2

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 1 recap step 2: completed")

### Lab 1 quick recap 3

Pause and summarize one takeaway from the previous cell before moving on.

## Final checkpoint

In [ ]:
assert len(df) == 1000
assert int(df["is_fraud"].sum()) == 10
assert amount_outliers == 58
assert distance_outliers == 58
assert abs(legit_amount_mean - 43.22) < 1.0
assert abs(fraud_amount_mean - 233.94) < 5.0
assert abs(max_fraud_distance - 51.72) < 1.0
print("Numbers match — you're good.")



## Reflection

Why are there more IQR outliers than fraud cases?